# Feature Engineering Research — What Else Could We Add?
## Concrete, buildable candidate features for VEP-nAChR

*Date: 2026-07-09*

**Purpose.** `NOTES2.ipynb` §9 and §11 already sketched a wishlist of structural and non-structural features (pore-axis geometry, interface distance, MSA conservation, ESM-2 embeddings, domain labels). This notebook turns that wishlist into **concrete, sourced, buildable recipes** — the actual tool/package to use, the effort level, and (per the circularity-tier convention introduced in `VEP_reserach.ipynb` §4) whether each feature is population-free, population-tuned, or clinical-trained. It draws on the literature scan done for `VEP_reserach.ipynb` plus a dedicated search for practical, no-training-required tooling.

**Framing constraint (unchanged from `vep_research.ipynb` §5): we have ~351 labeled samples.** Every recommendation below is either (a) a pretrained model used zero-shot / as a feature source, or (b) a classical, cheaply-computed descriptor — nothing here proposes training a new deep model from scratch on our data.

---
## 1. Current feature inventory (baseline, from `vep_nachr/config.py` and `structural.py`)

| Group | # features | Contents |
|---|---|---|
| Physicochemical | 24 | 8 AAIndex scales (hydrophobicity, polarity, volume, MW, charge, isoelectric point, aromaticity, helix propensity) × {wt, mt, diff} |
| Substitution | 3 | BLOSUM62 raw + normalized, Grantham distance |
| Positional | 17 | normalized position (1) + subunit one-hot (16) |
| Structural | 8 | RSA (Shrake-Rupley SASA), B-factor, DSSP helix/sheet/coil, C-beta density, half-sphere exposure up/down |
| **Total** | **52** | |

Best result so far (full nested-CV + Optuna, per `NOTES2.ipynb` §8b): **LogReg F1 = 0.660** on GOF, closely trailed by LightGBM/SVM/RF, all within noise of each other. The structural block, even after fixing the dead-RSA bug, sits at "present but not decisive" — the field-wide pattern (both VEP-Enac's p=0.80 finding and our own ablation) says **the next real gain is more likely to come from a new feature *category*, not more structural features from the same 4 PDB structures.**

**Circularity-tier convention** (from Livesey & Marsh 2025, see `VEP_reserach.ipynb` §4.2) is used below: **PF** = population-free (no exposure to human clinical/population labels), **PT** = population-tuned (seen allele-frequency data), **CT** = clinical-trained (trained on ClinVar/HGMD-style labels). All 52 current features are **PF** — worth stating explicitly, since some additions below are not.

---
## 2. Priority matrix (read this first)

| # | Feature | Category | Effort | Tier | Payoff (expected) |
|---|---|---|---|---|---|
| 3.1 | ESM-2 zero-shot log-likelihood-ratio (LLR) | pLM embedding/score | Low-Medium | PF | **High** — closest thing to "free" state-of-the-art signal |
| 3.2 | AlphaMissense score (bulk lookup) | pathogenicity score | Low | PT | Medium-High, but tag circularity |
| 3.3 | GEMME / iGEMME score | pathogenicity score | Low-Medium | PF | Medium — cheap, avoids PT tagging concerns |
| 3.4 | MSA conservation (ConSurf or DIY Shannon entropy) | evolutionary | Low (ConSurf) / Medium (DIY) | PF | **High** — the single biggest lever in classic VEP literature, and we currently have **none** of it |
| 3.5 | ThermoMPNN ΔΔG (structural stability) | structural | Medium | PF | Medium — LOF correlate |
| 3.6 | Domain/region indicator (ECD / M2 pore / ICD) | structural/categorical | Low | PF | Medium-High — mechanistically justified by the CMS paper |
| 3.7 | HPO phenotype-similarity kernel | phenotype | Medium | PF/CT-adjacent* | Medium, **novel** — nobody in the GOF/LOF literature uses this yet |
| 3.8 | Pore-axis radial distance + membrane depth (via HOLE) | structural/geometric | Medium | PF | Medium-High — nAChR-specific, mechanistically motivated |
| 3.9 | Interface distance + multi-radius contact count | structural | Low-Medium (extends existing code) | PF | Medium |
| 3.10 | Subunit phylogeny / MTL task-similarity kernel | meta/training-architecture | Medium-High | PF | Addresses the uneven-per-subunit-data problem directly |
| 3.11 | DMS-style continuous target (Envision framing) | training-target reframing | High (needs transfer data) | PF | Long-horizon, exploratory |
| 3.12 | Self-distillation (Born-Again Networks) | training technique | Medium | n/a (technique, not a feature) | Squeezes generalization from small data, no new labels needed |

*HPO annotations are drawn from patient phenotype records, so treat as "PF for the sequence/structure model, but audit separately for any clinical-label leakage if HPO terms were ever derived from the same case reports as our GOF/LOF labels."*

**Recommended build order:** 3.4 (conservation) and 3.6 (domain indicator) first — cheapest, most mechanistically justified, zero new dependencies beyond what's already in the repo. Then 3.1 (ESM-2) and 3.2/3.3 (score lookups) — all "add a column," no new training. Then 3.8-3.9 (finish the structural-geometry wishlist from `NOTES2.ipynb` §9). Treat 3.7, 3.10, 3.11, 3.12 as second-phase / stretch goals.

---
## 3.1 ESM-2 zero-shot log-likelihood-ratio (LLR)

**What it captures:** a protein language model's implicit sense of "how surprising is this substitution given everything the model has learned about protein sequences" — implicitly encodes conservation + structure + biophysics learned from millions of sequences, without needing an MSA of nAChR orthologs specifically.

**How, concretely:**
- The original `facebookresearch/esm` GitHub repo was **archived August 2024** — use **Hugging Face `transformers`** instead: `AutoTokenizer` + `EsmForMaskedLM`.
- **Masked-marginal scoring**: mask the mutated position in the wildtype sequence, run one forward pass, read off the logits, then `LLR = log P(mutant AA) - log P(wildtype AA)` at that position. A single masked forward pass per mutation (not full-sequence exhaustive masking) is standard practice and nearly as good per the ProteinGym literature.
- **Model size — bigger is not better here.** Per a 2025 *Scientific Reports* study on transfer learning with realistic (small) datasets, **ESM-2 150M or 650M** matches or beats the 3B/15B variants on DMS/variant-effect transfer tasks. Recommended: **`facebook/esm2_t33_650M_UR50D`** (fallback: `..._t12_35M...` or `..._t30_150M...` if compute-constrained).
- **Cost:** ~2.5GB one-time download; CPU-feasible for single-sequence zero-shot scoring (no fine-tuning, no batching needed) — minutes for all ~351 mutations across 16 subunit sequences.
- **Tier:** PF (pretrained on UniRef, no clinical/population labels).

---
## 3.2 & 3.3 Pathogenicity score lookups: AlphaMissense vs. GEMME/iGEMME

**AlphaMissense (3.2):**
- Full-proteome precomputed scores are a **bulk download, not a model run**: Zenodo record **10.5281/zenodo.8208688** → `AlphaMissense_aa_substitutions.tsv.gz` (~1.2GB, per-position/per-substitution scores for all human canonical UniProt proteins). Also queryable via Ensembl (integrated since May 2024), UniProt, AlphaFold DB, or the R/Bioconductor package `AlphaMissenseR`.
- Since it's proteome-wide, **every CHRNA/CHRNB/CHRND/CHRNE/CHRNG subunit should already have precomputed scores** — filter the TSV by UniProt accession (already in `config.py`'s `CANONICAL_ACCESSIONS`... note: those are RefSeq NP_ IDs, will need a RefSeq→UniProt cross-reference, already implicitly available since `PDB_MAPPING` chain assignments were done via UniProt sequences).
- **Tier: PT (population-tuned)** — fine-tuned using allele frequency as a weak label. Per the PIEZO1 case study (`VEP_reserach.ipynb` §3.3), still usable as a GOF/LOF-*correlated* signal when combined with structural context, but must be tagged, not treated as population-free.

**GEMME / iGEMME (3.3):**
- Population-free alternative: phylogeny-derived substitution-likelihood scoring, no allele-frequency exposure. iGEMME scales to large proteins and is far cheaper computationally than a protein language model.
- Requires building/obtaining an MSA of the target protein family (same alignment step needed for §3.4 conservation — can share the same MSA-building work).
- **Tier: PF.** Use alongside AlphaMissense specifically so we have at least one PT and one PF pathogenicity score to compare/ensemble, per the circularity-tier discipline in `VEP_reserach.ipynb` §4.

---
## 3.4 MSA-based conservation (the single biggest missing feature category)

We currently have **zero** conservation/evolutionary features — a notable gap, since conservation is the backbone of essentially every classical VEP (SIFT, PolyPhen-2) and both funNCion and LoGoFunc lean on it.

**Option A — ConSurf server (lowest effort):** submit each of the 16 wildtype FASTAs (already in the repo) to [consurf.tau.ac.il](https://consurf.tau.ac.il/overview.php). It automates BLAST/PSI-BLAST homolog collection → MAFFT/MUSCLE/CLUSTALW alignment → neighbor-joining phylogenetic tree → **Rate4Site** evolutionary-rate scoring per residue. Free, web-based, minutes-to-hours turnaround, no local pipeline to maintain.

**Option B — DIY MAFFT + Shannon entropy (medium effort, fully local/reproducible):** pull nAChR orthologs from UniProt across species (human/mouse/rat/chicken — and for muscle-type subunits, *Torpedo* nAChR has an unusually deep structural/sequence literature), align with `mafft --auto`, then compute per-column Shannon entropy directly in a few lines of Biopython/NumPy. Avoids ConSurf's Bayesian/tree machinery if full local reproducibility matters more than convenience.

**Either way, two derived features follow naturally and were already sketched in `NOTES2.ipynb` §11.2-3:**
- **Substitution likelihood against the PSSM column** (score the *specific* wt→mut change against what that position tolerates, not just position-agnostic BLOSUM62).
- **Grantham/Miyata deviation from the column consensus** (how chemically odd the variant is *relative to what this position accepts*, not just in the abstract).

**Tier: PF** in both options (no clinical/population label exposure — pure cross-species sequence evolution).

---
## 3.5 Structure-based ΔΔG (stability change)

**Recommended: ThermoMPNN** (`Kuhlman-Lab/ThermoMPNN` on GitHub) — a GNN transfer-learned from ProteinMPNN, ships pretrained weights, no training needed. Run `custom_inference.py` against our existing PDB/CIF structures to get per-position, per-substitution ΔΔG for all 19 possible mutants in one pass (site-saturation style) — a natural extension of the structural pipeline we already have (`structural.py` already loads and caches these same CIF files).

**Not recommended right now: RaSP** — also pretrained/no-training, but its Colab support has broken (dependency rot) and it needs a local conda env pinned to old Python 3.6/PyTorch versions. ThermoMPNN is the more maintainable choice today.

**Why this should help:** destabilizing mutations correlate with LOF (a misfolded/degraded channel can't function at all); a large *positive* ΔΔG (destabilizing) alongside a pore-lining location could help disambiguate LOF-via-misfolding from LOF-via-altered-gating.

**Coverage caveat:** same limitation as our existing structural features — only computable for the 10 subunits with a PDB/AlphaFold structure loaded; the 6 uncovered subunits (`NOTES2.ipynb` §10 table) would need AlphaFold models fetched first, same prerequisite already identified there.

**Tier: PF** (trained on ProteinMPNN's structural/sequence objective, no clinical labels).

---
## 3.6 Structural domain/region indicator (ECD vs. M2 pore vs. ICD)

Already sketched in `NOTES2.ipynb` §11.5 as a "cheap and interpretable" idea; now has direct mechanistic backing from the CMS paper (`VEP_reserach.ipynb` §3.4): **LOF mutations cluster in the extracellular ligand-binding domain (binding-site loops), GOF mutations cluster in the M2 pore-lining helix.** This is arguably the most directly-motivated new feature in this whole notebook, because it's backed by a mechanistic paper *about this exact protein family*, not an inference from a different channel.

**Implementation:** derive from UniProt sequence-feature annotations (topological domain / transmembrane region records already available per subunit) — categorical label per residue: `{ECD, TM1, TM2 (pore-lining), TM3, TM4, ICD}`. Cheap, no new external tool, reuses data already on hand (the UniProt FASTAs + their feature tables).

**Tier: PF.**

---
## 3.7 HPO phenotype-similarity kernel (novel — not used by any GOF/LOF tool we found)

From the ion-channel phenotypic ML paper (`VEP_reserach.ipynb` §3.1): a multiple-kernel-learning framework where a **Human Phenotype Ontology (HPO) semantic-similarity kernel** (computed via Jaccard, Lin, or Resnik similarity between the phenotype terms associated with a variant's case report and reference phenotype sets for e.g. congenital myasthenic syndrome / ADNFLE / nicotine-dependence) is combined with sequence/structural features.

**Why it's worth the extra effort:** none of funNCion/LoGoFunc/PreMode/MissION use this. nAChR mutations are unusually well phenotype-annotated (three distinct clinical contexts: CMS, epilepsy, nicotine-dependence GWAS), so the raw material for this kernel likely already exists in our source literature/case reports — it just hasn't been extracted as a feature yet.

**Implementation sketch:** (1) map each mutation's associated pathology (already a raw data column per `config.py`'s `COLUMN_MAPPING`, `"Pathology" → "pathology"`) to HPO terms, (2) compute semantic similarity to a small set of reference phenotype clusters, (3) use the similarity scores as features. This is a genuinely new research contribution, not just an engineering add — flagged as such in `VEP_reserach.ipynb` §5 point 7.

**Tier:** phenotype-derived, so treat as **PF for the sequence/structure model itself**, but audit for label leakage if HPO terms and GOF/LOF labels were ever extracted from the *same* case-report sentence (a realistic risk worth a specific leakage check before use).

---
## 3.8 Pore-axis radial distance + membrane depth (via HOLE, not ad hoc PCA)

`NOTES2.ipynb` §9 (Tier 1, #1-2) already proposed this as the highest-value nAChR-specific structural feature, suggesting "PCA on Cα coordinates" to find the pore axis. The literature scan found the field-standard tool for this instead:

- **HOLE** (classic pore-radius program, still the standard) computes pore radius as a function of position along the channel's central axis from a static PDB structure. Has a modern Python wrapper via **`MDAnalysis.analysis.hole2`** — usable directly on our existing cryo-EM structures (7QKO, 7EKI, 6CNJ, 6PV7) without any MD simulation, just the static coordinates already loaded by `structural.py`.
- For trajectory-based / hydrophobic-gate detection (more rigorous but heavier), **CHAP** (Channel Annotation Package, Klesse et al., GROMACS-based, channotation.org) extends HOLE with hydrophobicity + water-density profiling and the standard computational definition of a "hydrophobic gate" (radius ≤ ~4Å + hydrophobic lining → dewetting/functional closure). This needs an MD trajectory to be fully rigorous, but a single-frame static run is feasible as a lighter-weight approximation.
- **Recommendation:** use `MDAnalysis.analysis.hole2` on the existing static structures to get the pore-axis coordinate and radius profile — this *is* the field's actual definition of "pore axis," rather than an ad hoc geometric centroid — then derive:
  1. **Radial distance** = residue's perpendicular distance to the HOLE-derived axis (small = pore-facing — the textbook GOF/LOF hotspot zone).
  2. **Axial position / membrane depth** = residue's projection onto that same axis (a quantitative ECD/TMD/ICD coordinate, replacing the current crude `position_normalized` = sequence-index/max).

**Caveat carried over from `NOTES2.ipynb` §9:** these features require the *assembled pentamer* — available for the 4 experimental cryo-EM structures, but not for AlphaFold *monomer* models of the 6 uncovered subunits (no quaternary context) unless those monomers are computationally assembled into a pentamer first.

**Tier: PF.**

---
## 3.9 Interface distance + multi-radius contact count (extends existing code, low incremental effort)

Both already proposed in `NOTES2.ipynb` §9 (Tier 2, #3-4) and unchanged by this literature pass — listed here only to keep the full roadmap in one place:

- **Interface proximity**: minimum distance from a residue to any atom of a *different* chain — relevant because the ACh binding site and many gating-critical residues sit at subunit-subunit interfaces.
- **Multi-radius contact number**: generalize the existing `cbeta_density` feature to multiple radii (e.g. 8Å and 12Å), optionally split same-chain vs. cross-chain, for a richer packing/burial signal.

Both reuse the same cached-structure pattern already established in `structural.py` (Shrake-Rupley SASA, HSE) — lowest-incremental-effort item on this whole list since the CIF-loading and neighbor-search machinery already exists.

**Tier: PF.**

---
## 3.10 Subunit phylogeny / multi-task-learning kernel (architecture-level, not a per-row feature)

From the K⁺-channel MTL paper (`VEP_reserach.ipynb` §3.2): instead of a single classifier trained on all subunits pooled together (current approach, using subunit one-hot as a feature), build a **phylogeny/sequence-distance kernel between the 16 CHRN subunits** and use it in a multi-task-learning framework, so that data-rich subunits (CHRNA1, CHRNE — both CMS-relevant) can lend statistical strength to data-poor ones (CHRNA6: 18 imputed rows, CHRNA2: 9, CHRNA5: 5, CHRNB3: 4, CHRNA9/A10: 2 each, per `NOTES2.ipynb` §10).

This is a bigger architectural change than the other items here (not just "add a column" — it changes the training setup), so it's marked medium-high effort and positioned as a second-phase item, but it directly targets the exact data-imbalance problem this project already knows it has.

**Tier: n/a (architecture, not a feature per se) — PF if the kernel is built from sequence/phylogenetic distance alone.**

---
## 3.11 & 3.12 Two longer-horizon ideas

**3.11 — DMS-style continuous target (Envision framing).** Envision (`VEP_reserach.ipynb` §2) trains on deep-mutational-scanning continuous fitness scores rather than binary labels. No nAChR-specific DMS dataset exists (confirmed absent from the Livesey & Marsh 36-protein set and, as far as this scan found, MaveDB) — but if a homologous ion-channel DMS dataset can be found, it could serve as **transfer/pretraining signal** or a sanity-check target, since continuous scores carry more direction/magnitude information than a binary GOF/LOF label. Flagged as exploratory, not immediately actionable.

**3.12 — Self-distillation (Born-Again Networks, from MTBAN, `VEP_reserach.ipynb` §2).** A training-time technique, not a feature: retrain a model on its own prior generation's soft outputs to improve generalization *without new labels*. Worth prototyping only if/when we fine-tune an embedding-based model (e.g. ESM) on the nAChR data directly, rather than using it purely zero-shot as in §3.1 — squeezes extra generalization out of a small labeled set at effectively no extra data cost.

---
## References and links

*Same source set as `VEP_reserach.ipynb` §References — repeated here for the specific tool/package citations relevant to feature implementation.*

- ESM-2 / Hugging Face `transformers` docs: https://huggingface.co/docs/transformers/en/model_doc/esm — model IDs `facebook/esm2_t12_35M_UR50D` through `..._t36_3B...`.
- ESM-2 model-size-vs-transfer-learning study — *Scientific Reports* (2025), "Medium-sized protein language models perform well at transfer learning on realistic datasets," PMC11601519.
- AlphaMissense bulk data — Zenodo 10.5281/zenodo.8208688; R/Bioconductor `AlphaMissenseR`.
- GEMME / iGEMME, ESCOTT, popEVE, CPT-1, SaProt, VARITY — see `VEP_reserach.ipynb` §2 (Livesey & Marsh 2025, *Genome Biology*) for full citations.
- ConSurf server: https://consurf.tau.ac.il/overview.php (Rate4Site evolutionary-rate scoring).
- ThermoMPNN: https://github.com/Kuhlman-Lab/ThermoMPNN
- RaSP: Blaabjerg LM, et al. (2023). *eLife* 12:e82593 — noted environment/Colab fragility, ThermoMPNN preferred.
- HOLE / `MDAnalysis.analysis.hole2`: https://docs.mdanalysis.org/stable/documentation_pages/analysis/hole2.html
- CHAP (Channel Annotation Package) — Klesse et al., channotation.org; hydrophobic-gate heuristic, PMC6628796.
- Ion-channel phenotypic ML (HPO/MKL) and voltage-gated K+ channel MTL papers — see `VEP papers/` folder and `VEP_reserach.ipynb` §3.1-3.2.
- Engel, Ohno & Sine (2002), CMS mechanistic paper — see `VEP_reserach.ipynb` §3.4.
- Envision — Gray et al. (2018), *Cell Systems*.

**See also:** `VEP_reserach.ipynb` for the model-landscape context these features are meant to support, and `NOTES2.ipynb` §9-11 for this project's original feature wishlist (largely superseded/made concrete here) and the current 52-feature baseline this notebook extends.